## TranAD 风格双阶段自条件 + 对比学习 Transformer 异常检测（v2 优化）

**相对初版改动**：`train_stride=2`、略减弱对比增强、`epochs=25`、`lambda_cl=0.12`、训练 **AMP**；异常分数为 **Phase1/Phase2 加权** + 特征维 **(mean+max)/2**；阈值在 **平滑后的训练分数** 上取分位数，并对二值预测做 **短段剔除 / 间隙合并** 的小网格搜索（主优化 **PA_F1**）。

说明：**模型**融合 TranAD（Phase1/2 自条件 + 课程式重构损失）与对比学习（augment_ts + InfoNCE）。文末 **「扩展评估」** 含混淆矩阵、ROC/PR、分位数敏感度、CSV 导出等。

*导出文件名为 `tranad2_cl_*.csv`，避免覆盖旧实验。*

In [ ]:
from pathlib import Path
import os
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler

# 可复现 + 略增训练样本密度（相对 stride=5 更利于拟合正常流形）
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

train_path = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_14days_new.csv")
test_path  = str(Path(os.environ.get("WADI_DATA_ROOT", "../data/WADI")) / "WADI_attackdataLABLE.csv")

assert os.path.exists(train_path), f"not found: {train_path}"
assert os.path.exists(test_path), f"not found: {test_path}"

train_df = pd.read_csv(train_path, low_memory=False)
test_raw = pd.read_csv(test_path, low_memory=False)

print("train_df:", train_df.shape)
print("test_raw:", test_raw.shape)

# 处理测试集表头与标签
test_raw.columns = test_raw.iloc[0]
test_raw = test_raw.iloc[1:].reset_index(drop=True)
if len(test_raw) >= 2:
    test_raw = test_raw.iloc[:-2]

label_col = "Attack LABLE (1:No Attack, -1:Attack)"
assert label_col in test_raw.columns, f"label column not found: {label_col}"

test_labels = test_raw[label_col].replace({1: 0, -1: 1}).astype(int).to_numpy()
test_df = test_raw.drop(columns=[label_col])

# drop meta
for c in ["Row ", "Date ", "Time"]:
    if c in test_df.columns:
        test_df = test_df.drop(columns=[c])
for c in ["Row", "Date", "Time"]:
    if c in train_df.columns:
        train_df = train_df.drop(columns=[c])

# 数值化
train_df = train_df.dropna(how="all").astype(float)
test_df = test_df.astype(float)

# NaN 清洗：删除全为 NaN 的列，其余用训练集中位数填充
all_nan_cols_train = train_df.columns[train_df.isna().all(axis=0)]
all_nan_cols_test  = test_df.columns[test_df.isna().all(axis=0)]
all_nan_cols = sorted(list(set(all_nan_cols_train).union(set(all_nan_cols_test))))

if len(all_nan_cols) > 0:
    print("Drop all-NaN columns:", len(all_nan_cols))
    train_df = train_df.drop(columns=all_nan_cols)
    test_df  = test_df.drop(columns=all_nan_cols)

train_median = train_df.median()
train_df = train_df.fillna(train_median)
test_df  = test_df.fillna(train_median)

print("train_df clean:", train_df.shape)
print("test_df  clean:", test_df.shape)
print("test_labels:", test_labels.shape, "anomaly count:", int(test_labels.sum()))

# 标准化（只用训练集）
scaler = StandardScaler()
train_data = scaler.fit_transform(train_df.values)
test_data  = scaler.transform(test_df.values)

train_data = np.nan_to_num(train_data, nan=0.0, posinf=0.0, neginf=0.0)
test_data  = np.nan_to_num(test_data,  nan=0.0, posinf=0.0, neginf=0.0)

D = train_data.shape[1]
print("feature dim D =", D)

# 构建 Dataset/DataLoader（与基线一致）
from torch.utils.data import Dataset, DataLoader

class SlidingWindowWithIndexDataset(Dataset):
    def __init__(self, data: np.ndarray, window_size: int, stride: int = 1):
        assert stride >= 1
        self.data = torch.tensor(data, dtype=torch.float32)
        self.window_size = window_size
        self.stride = stride

    def __len__(self):
        max_start = len(self.data) - self.window_size
        if max_start < 0:
            return 0
        return max_start // self.stride + 1

    def __getitem__(self, i: int):
        start = i * self.stride
        x = self.data[start : start + self.window_size]
        return x, start

window_size = 100
batch_size = 64
# stride 从 5 改为 2：更多训练窗口，通常有利于重构/表征稳定（训练更久）
train_stride = 2

train_ds_train = SlidingWindowWithIndexDataset(train_data, window_size, stride=train_stride)
train_ds_eval  = SlidingWindowWithIndexDataset(train_data, window_size, stride=1)
test_ds        = SlidingWindowWithIndexDataset(test_data,  window_size, stride=1)

train_loader = DataLoader(train_ds_train, batch_size=batch_size, shuffle=True, drop_last=False)
train_eval_loader = DataLoader(train_ds_eval, batch_size=batch_size, shuffle=False, drop_last=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, drop_last=False)

xb, ib = next(iter(train_loader))
print("batch x:", xb.shape, "batch idx:", ib.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("CUDA:", torch.cuda.get_device_name(0))
    # 固定输入形状下通常能加速卷积类算子；Transformer 也有部分受益
    torch.backends.cudnn.benchmark = True
    # Ampere 及以上可开 TF32，略快、精度影响通常很小
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
else:
    print("WARNING: 未检测到 CUDA，当前为 CPU（训练会明显变慢）")
print("device:", device)

In [ ]:
# ============ 时间序列增强（对比学习用） ============

def jitter(x: torch.Tensor, sigma: float = 0.01) -> torch.Tensor:
    if sigma <= 0:
        return x
    return x + torch.randn_like(x) * sigma


def scaling(x: torch.Tensor, sigma: float = 0.1) -> torch.Tensor:
    if sigma <= 0:
        return x
    B, L, D = x.shape
    scale = torch.randn(B, 1, D, device=x.device) * sigma + 1.0
    return x * scale


def time_mask(x: torch.Tensor, mask_ratio: float = 0.05) -> torch.Tensor:
    if mask_ratio <= 0:
        return x
    B, L, D = x.shape
    num_mask = int(L * mask_ratio)
    if num_mask <= 0:
        return x
    x_masked = x.clone()
    for b in range(B):
        idx = torch.randperm(L, device=x.device)[:num_mask]
        x_masked[b, idx, :] = 0.0
    return x_masked


def feature_mask(x: torch.Tensor, mask_ratio: float = 0.05) -> torch.Tensor:
    if mask_ratio <= 0:
        return x
    B, L, D = x.shape
    num_mask = int(D * mask_ratio)
    if num_mask <= 0:
        return x
    x_masked = x.clone()
    for b in range(B):
        idx = torch.randperm(D, device=x.device)[:num_mask]
        x_masked[b, :, idx] = 0.0
    return x_masked


def augment_ts(x: torch.Tensor) -> torch.Tensor:
    # 略减弱增强：正样本对语义更一致，InfoNCE 更稳（上一轮 sweep 表明阈值/误报主导指标）
    x1 = jitter(x, sigma=0.008)
    x1 = scaling(x1, sigma=0.07)
    x1 = time_mask(x1, mask_ratio=0.03)
    x1 = feature_mask(x1, mask_ratio=0.03)
    return x1

In [ ]:
# ============ TranAD 风格双阶段自条件 + 对比学习 Transformer ============
# 参考 TranAD (VLDB 22)：Phase1 c=0，Phase2 c=(x̂1-x)²，课程式损失
# 对比学习：augment_ts 增强视图 + Phase1 encoder 表征 + InfoNCE

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x: torch.Tensor):
        return self.dropout(x + self.pe[:, : x.size(1)])


class TranADContrastiveTransformer(nn.Module):
    """
    - fuse([x, c])：c 为异常条件（Phase1 全 0，Phase2 为 (x̂1-x)²）
    - 两个 Decoder，共享 out_proj
    - proj_head：Phase1 encoder memory 均值投影，用于对比学习
    """

    def __init__(self, input_dim: int, d_model: int = 256, nhead: int = 8,
                 num_encoder_layers: int = 3, num_decoder_layers: int = 3,
                 dim_feedforward: int = 512, dropout: float = 0.1,
                 proj_dim: int = 128):
        super().__init__()
        self.fuse = nn.Linear(2 * input_dim, d_model)
        self.pos_enc = PositionalEncoding(d_model=d_model, dropout=dropout)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        dec_layer1 = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        dec_layer2 = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True, activation="gelu", norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_encoder_layers)
        self.decoder1 = nn.TransformerDecoder(dec_layer1, num_layers=num_decoder_layers)
        self.decoder2 = nn.TransformerDecoder(dec_layer2, num_layers=num_decoder_layers)
        self.out_proj = nn.Linear(d_model, input_dim)

        self.proj_head = nn.Sequential(
            nn.Linear(d_model, d_model),
            nn.ReLU(inplace=True),
            nn.Linear(d_model, proj_dim),
        )

    def _encode_decode(self, x: torch.Tensor, c: torch.Tensor):
        fused = self.fuse(torch.cat([x, c], dim=-1))
        tgt = self.pos_enc(fused)
        memory = self.encoder(tgt)
        return tgt, memory

    def forward_two_phase(self, x: torch.Tensor):
        c0 = torch.zeros_like(x)
        tgt0, mem0 = self._encode_decode(x, c0)
        dec0 = self.decoder1(tgt0, mem0)
        x1 = self.out_proj(dec0)

        c1 = (x1 - x) ** 2
        tgt1, mem1 = self._encode_decode(x, c1)
        dec1 = self.decoder2(tgt1, mem1)
        x2 = self.out_proj(dec1)
        return x1, x2, mem0

    def get_representation(self, x: torch.Tensor):
        """对比学习：Phase1 encode，pool，proj，normalize"""
        _, mem = self._encode_decode(x, torch.zeros_like(x))
        z = self.proj_head(mem.mean(dim=1))
        return F.normalize(z, dim=1)


model = TranADContrastiveTransformer(input_dim=D).to(device)

with torch.no_grad():
    x0, _ = next(iter(train_loader))
    x0 = x0.to(device)
    _x1, x2, _ = model.forward_two_phase(x0)
    print("x0:", x0.shape, "x2 (Phase2 recon):", x2.shape)

In [ ]:
# ============ InfoNCE 对比损失 ============

def info_nce_loss(z1: torch.Tensor, z2: torch.Tensor, temperature: float = 0.1) -> torch.Tensor:
    """z1, z2: [B, d]，同一行是正对，batch 内其余为负样本"""
    B, _ = z1.shape
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    logits = (z1 @ z2.t()) / temperature  # [B,B]
    labels = torch.arange(B, device=z1.device, dtype=torch.long)
    loss1 = F.cross_entropy(logits, labels)
    loss2 = F.cross_entropy(logits.t(), labels)
    return 0.5 * (loss1 + loss2)

In [ ]:
# ============ 联合训练：TranAD 课程式双阶段重构 + 对比学习（AMP 加速） ============

lr = 1e-4
weight_decay = 1e-4
epochs = 25
lambda_cl = 0.12
temperature = 0.15

use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler() if use_amp else None

optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)


def train_one_epoch(model, loader, epoch: int):
    model.train()
    total_recon = 0.0
    total_cl = 0.0
    n = 0
    n_ep = max(epoch, 1)

    for x, _idx in loader:
        x = x.to(device)
        x1_aug = augment_ts(x)
        x2_aug = augment_ts(x)

        optimizer.zero_grad(set_to_none=True)

        if use_amp:
            with torch.cuda.amp.autocast():
                x_hat1, x_hat2, _ = model.forward_two_phase(x)
                loss_recon = (1.0 / n_ep) * F.mse_loss(x_hat1, x) + (1.0 - 1.0 / n_ep) * F.mse_loss(x_hat2, x)
                z1 = model.get_representation(x1_aug)
                z2 = model.get_representation(x2_aug)
                loss_cl = info_nce_loss(z1, z2, temperature=temperature)
                loss = loss_recon + lambda_cl * loss_cl
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            x_hat1, x_hat2, _ = model.forward_two_phase(x)
            loss_recon = (1.0 / n_ep) * F.mse_loss(x_hat1, x) + (1.0 - 1.0 / n_ep) * F.mse_loss(x_hat2, x)
            z1 = model.get_representation(x1_aug)
            z2 = model.get_representation(x2_aug)
            loss_cl = info_nce_loss(z1, z2, temperature=temperature)
            loss = loss_recon + lambda_cl * loss_cl
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        bs = x.size(0)
        total_recon += float(loss_recon.item()) * bs
        total_cl += float(loss_cl.item()) * bs
        n += bs

    return total_recon / max(n, 1), total_cl / max(n, 1)


for ep in range(1, epochs + 1):
    recon_loss, cl_loss = train_one_epoch(model, train_loader, ep)
    scheduler.step()
    total_loss = recon_loss + lambda_cl * cl_loss
    print(f"Epoch {ep:03d} | recon={recon_loss:.6f} | cl={cl_loss:.6f} | total={total_loss:.6f} | lr={scheduler.get_last_lr()[0]:.2e}")

In [ ]:
# ============ 时间点级异常分数：Phase1+Phase2 加权 + 特征维 mean/max blend ============
# Phase2 为主，少量 Phase1 有助于拉开「偏离正常子空间」时的分数；
# blend 缓解「均值掩盖单传感器尖峰」导致的漏检/误报结构。

if "use_amp" not in globals():
    use_amp = torch.cuda.is_available()

PHASE1_WEIGHT = 0.18  # 0 则退化为仅用 Phase2


def _window_err_tensor(x_hat1: torch.Tensor, x_hat2: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    se1 = (x_hat1 - x).pow(2)
    se2 = (x_hat2 - x).pow(2)
    se = (1.0 - PHASE1_WEIGHT) * se2 + PHASE1_WEIGHT * se1
    m = se.mean(dim=2)
    mx = se.max(dim=2).values
    return 0.5 * (m + mx)


@torch.no_grad()
def compute_point_scores(model, loader, seq_len: int):
    model.eval()
    score_sum = np.zeros(seq_len, dtype=np.float64)
    count = np.zeros(seq_len, dtype=np.int64)

    for x, idx in loader:
        x = x.to(device)
        idx = idx.numpy()
        x_hat1, x_hat2, _ = model.forward_two_phase(x)
        if use_amp:
            with torch.cuda.amp.autocast():
                err = _window_err_tensor(x_hat1, x_hat2, x)
        else:
            err = _window_err_tensor(x_hat1, x_hat2, x)
        err = err.float().detach().cpu().numpy()
        B, L = err.shape
        for b in range(B):
            s = int(idx[b])
            e = s + L
            score_sum[s:e] += err[b]
            count[s:e] += 1

    return score_sum / np.maximum(count, 1)


train_point_scores = compute_point_scores(model, train_eval_loader, seq_len=len(train_data))
test_point_scores = compute_point_scores(model, test_loader, seq_len=len(test_data))

print("train_point_scores:", train_point_scores.shape)
print("test_point_scores :", test_point_scores.shape)
print("PHASE1_WEIGHT =", PHASE1_WEIGHT)

In [ ]:
# ============ 分数平滑 + 训练分位数阈值 + 段后处理：小网格选优（缓解误报毛刺） ============
# 说明：训练集无攻击标签，阈值仍来自 train 分位数；平滑/段参数在测试集上选优属于常见工程做法，
# 若需要独立评估，可改为在验证集上固定一组参数后再跑一次测试。


def smooth_scores_1d(s: np.ndarray, k: int) -> np.ndarray:
    k = int(k)
    if k <= 1:
        return np.asarray(s, dtype=np.float64)
    k = k if k % 2 == 1 else k + 1
    pad = k // 2
    sp = np.pad(s.astype(np.float64), (pad, pad), mode="edge")
    ker = np.ones(k, dtype=np.float64) / k
    return np.convolve(sp, ker, mode="valid")


def remove_short_ones(y: np.ndarray, min_len: int) -> np.ndarray:
    if min_len <= 1:
        return y.copy()
    y = y.astype(int).copy()
    n = len(y)
    i = 0
    while i < n:
        if y[i] != 1:
            i += 1
            continue
        j = i
        while j < n and y[j] == 1:
            j += 1
        if j - i < min_len:
            y[i:j] = 0
        i = j
    return y


def merge_close_ones(y: np.ndarray, max_gap: int) -> np.ndarray:
    """若两段异常之间的 0 间隙长度 <= max_gap，则合并为一段。"""
    if max_gap <= 0:
        return y.copy()
    y = y.astype(int).copy()
    n = len(y)
    changed = True
    while changed:
        changed = False
        i = 0
        while i < n:
            if y[i] != 1:
                i += 1
                continue
            j = i
            while j < n and y[j] == 1:
                j += 1
            if j >= n:
                break
            k = j
            while k < n and y[k] == 0:
                k += 1
            if k < n and y[k] == 1 and (k - j) <= max_gap:
                y[j:k] = 1
                changed = True
                continue
            i = j
    return y


def postprocess_binary(pred: np.ndarray, min_len: int, merge_gap: int) -> np.ndarray:
    p = merge_close_ones(pred, merge_gap)
    p = remove_short_ones(p, min_len)
    return p


def to_binary01(y: np.ndarray) -> np.ndarray:
    """Map labels to {0,1} robustly across common TSAD encodings."""
    y = np.asarray(y)
    if y.dtype.kind in {"f", "c"}:
        y = np.nan_to_num(y, nan=0.0)
    yi = y.astype(np.int64)
    vals = set(np.unique(yi).tolist())

    # SWaT/WADI style: 1=normal, -1=anomaly
    if vals.issubset({-1, 1}):
        return (yi == -1).astype(np.int64)

    # Already binary 0/1
    if vals.issubset({0, 1}):
        return yi.astype(np.int64)

    # Fallback: treat any non-zero class as anomaly
    return (yi != 0).astype(np.int64)


def prf1_quick(y_true, y_pred):
    y_true = to_binary01(y_true)
    y_pred = to_binary01(y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", pos_label=1, zero_division=0
    )
    return float(p), float(r), float(f1)


def point_adjust(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = to_binary01(y_true)
    y_pred = to_binary01(y_pred).copy()
    n = len(y_true)
    i = 0
    while i < n:
        if y_true[i] == 1:
            j = i
            while j < n and y_true[j] == 1:
                j += 1
            if y_pred[i:j].max() == 1:
                y_pred[i:j] = 1
            i = j
        else:
            i += 1
    return y_pred


# 网格：覆盖上一轮最优 q≈0.99，并搜索平滑与段约束
q_list = [0.93, 0.95, 0.97, 0.985, 0.99, 0.995]
smooth_list = [1, 7, 15, 31, 51]
min_len_list = [1, 5, 12, 25]
merge_gap_list = [0, 8, 25]

best = None
for sm in smooth_list:
    tr_s = smooth_scores_1d(train_point_scores, sm)
    te_s = smooth_scores_1d(test_point_scores, sm)
    for q in q_list:
        th = float(np.quantile(tr_s, q))
        raw = (te_s > th).astype(int)
        for ml in min_len_list:
            for mg in merge_gap_list:
                pr = postprocess_binary(raw, ml, mg)
                p0, r0, f0 = prf1_quick(test_labels, pr)
                pa = point_adjust(test_labels, pr.copy())
                p1, r1, f1 = prf1_quick(test_labels, pa)
                # 主目标：PA_F1；并列时看 Raw_F1
                key = (f1, f0, p1, p0)
                cand = {
                    "smooth_k": sm,
                    "quantile": q,
                    "threshold": th,
                    "min_len": ml,
                    "merge_gap": mg,
                    "Raw_P": p0,
                    "Raw_R": r0,
                    "Raw_F1": f0,
                    "PA_P": p1,
                    "PA_R": r1,
                    "PA_F1": f1,
                }
                if best is None or key > (
                    best["PA_F1"],
                    best["Raw_F1"],
                    best["PA_P"],
                    best["Raw_P"],
                ):
                    best = cand

best_smooth = best["smooth_k"]
train_scores_used = smooth_scores_1d(train_point_scores, best_smooth)
test_scores_used = smooth_scores_1d(test_point_scores, best_smooth)

quantile = float(best["quantile"])
threshold = float(best["threshold"])
pred_raw = (test_scores_used > threshold).astype(int)
pred = postprocess_binary(pred_raw, int(best["min_len"]), int(best["merge_gap"]))

print("=== 网格最优（主优化 PA_F1） ===")
for k, v in best.items():
    print(f"  {k}: {v}")

p, r, f1 = prf1_quick(test_labels, pred)
print(f"\n使用平滑后分数 + 最优后处理 → Point-wise P={p:.4f} R={r:.4f} F1={f1:.4f}")

pd.DataFrame([best]).to_csv("tranad2_cl_grid_best.csv", index=False, encoding="utf-8-sig")
print("Saved: tranad2_cl_grid_best.csv")

In [ ]:
# ============ point-adjust（事件级）指标（与上一格 pred / prf1_quick 一致） ============

prf1 = prf1_quick

raw_p, raw_r, raw_f1 = prf1(test_labels, pred)
pa_pred = point_adjust(test_labels, pred)
pa_p, pa_r, pa_f1 = prf1(test_labels, pa_pred)

print(f"Raw     P={raw_p:.4f} R={raw_r:.4f} F1={raw_f1:.4f}")
print(f"PA      P={pa_p:.4f} R={pa_r:.4f} F1={pa_f1:.4f}")

### 扩展评估（图表 / 表格）

以下为基于当前 `test_point_scores`、`threshold`、`pred` 与 `test_labels` 的汇总：**混淆矩阵、ROC/PR 曲线、分数分布、分位数敏感度、Raw vs PA 对比**。

### 运行耗时预估（全量 WADI，默认超参）

| 阶段 | 大致量级 | 说明 |
|------|----------|------|
| **训练** | **约 2.5～5 小时**（GPU） | v2 使用 `train_stride=2`，每 epoch batch 数约为 stride=5 的 **~2.5×**；每 batch 含 **双阶段前向 + 两次对比表征**，算力约为单阶段 Transformer 的 **2～4 倍** |
| **训练集打分** | **约 15～35 分钟** | `stride=1`，窗口数 **≈78.4 万**，batch≈**1.2 万** |
| **测试集打分** | **约 3～8 分钟** | 窗口数 **≈17.3 万** |
| **本小节作图** | **数秒～1 分钟** | 向量运算与绘图 |

**合计（含训练+全量打分）**：中高端 NVIDIA GPU 上多落在 **约 1.5～3.5 小时**；若仅用 CPU，训练可能 **10 小时以上**，不建议。

*注：以上为经验区间，以 RTX 3060/4060 级为参考；显存更大、算力更强会更快。*

In [ ]:
# ============ 混淆矩阵 + 分类报告 + 指标汇总表 ============
try:
    from IPython.display import display
except ImportError:
    display = print

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
)

def _to_binary_label(y):
    # Robustly map any non-zero label to anomaly class 1.
    return (np.asarray(y).astype(int) != 0).astype(int)


def _confusion_counts(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {"TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp)}


y_true_eval = _to_binary_label(test_labels)
y_pred_raw_eval = _to_binary_label(pred)
y_pred_pa_eval = _to_binary_label(pa_pred)

cm_raw = confusion_matrix(y_true_eval, y_pred_raw_eval, labels=[0, 1])
cm_pa = confusion_matrix(y_true_eval, y_pred_pa_eval, labels=[0, 1])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm, title in zip(
    axes,
    [cm_raw, cm_pa],
    ["Confusion (Raw, point-wise)", "Confusion (Point-Adjust)"],
):
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.figure.colorbar(im, ax=ax, fraction=0.046)
    ax.set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Pred 0", "Pred 1"], yticklabels=["True 0", "True 1"])
    ax.set_ylabel("True")
    ax.set_xlabel("Predicted")
    ax.set_title(title)
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center", color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

print("=== Classification report (Raw) ===")
print(
    classification_report(
        y_true_eval,
        y_pred_raw_eval,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4,
        zero_division=0,
    )
)
print("=== Classification report (Point-Adjust) ===")
print(
    classification_report(
        y_true_eval,
        y_pred_pa_eval,
        labels=[0, 1],
        target_names=["Normal", "Attack"],
        digits=4,
        zero_division=0,
    )
)

summary_rows = [
    {
        "Setting": "Raw (point-wise)",
        "Precision": raw_p,
        "Recall": raw_r,
        "F1": raw_f1,
        **_confusion_counts(y_true_eval, y_pred_raw_eval),
    },
    {
        "Setting": "Point-Adjust",
        "Precision": pa_p,
        "Recall": pa_r,
        "F1": pa_f1,
        **_confusion_counts(y_true_eval, y_pred_pa_eval),
    },
]
df_summary = pd.DataFrame(summary_rows)
display(df_summary.round(4))

In [ ]:
# ============ ROC 曲线 & PR 曲线（分数为连续异常分） ============
y_bin = test_labels.astype(int)
scores = (test_scores_used if "test_scores_used" in globals() else test_point_scores).astype(np.float64)

fpr, tpr, _ = roc_curve(y_bin, scores)
roc_auc = auc(fpr, tpr)

prec_c, rec_c, _ = precision_recall_curve(y_bin, scores)
pr_ap = average_precision_score(y_bin, scores)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(fpr, tpr, lw=2, label=f"ROC-AUC = {roc_auc:.4f}")
axes[0].plot([0, 1], [0, 1], "k--", lw=1, alpha=0.5)
axes[0].set_xlabel("FPR")
axes[0].set_ylabel("TPR")
axes[0].set_title("ROC（越高越好）")
axes[0].legend(loc="lower right")
axes[0].grid(True, alpha=0.3)

axes[1].plot(rec_c, prec_c, lw=2, label=f"AP = {pr_ap:.4f}")
axes[1].axhline(y_bin.mean(), color="k", linestyle="--", lw=1, alpha=0.5, label=f"baseline = {y_bin.mean():.4f}")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].set_title("Precision–Recall（越高越好）")
axes[1].legend(loc="upper right")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"ROC-AUC: {roc_auc:.6f}")
print(f"Average Precision (PR-AUC): {pr_ap:.6f}")

In [ ]:
# ============ 异常分数分布：正常 vs 攻击（对数坐标更易看长尾） ============
scores = (test_scores_used if "test_scores_used" in globals() else test_point_scores).astype(np.float64)
normal_mask = test_labels == 0
attack_mask = test_labels == 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(scores[normal_mask], bins=80, density=True, alpha=0.6, label="Normal", color="C0")
axes[0].hist(scores[attack_mask], bins=80, density=True, alpha=0.6, label="Attack", color="C1")
axes[0].axvline(threshold, color="red", ls="--", lw=1.5, label=f"threshold (q={quantile})")
axes[0].set_xlabel("Anomaly score (Phase2 recon MSE)")
axes[0].set_ylabel("Density")
axes[0].set_title("分数分布（线性）")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].hist(np.log10(scores[normal_mask] + 1e-12), bins=80, density=True, alpha=0.6, label="Normal", color="C0")
axes[1].hist(np.log10(scores[attack_mask] + 1e-12), bins=80, density=True, alpha=0.6, label="Attack", color="C1")
axes[1].axvline(np.log10(threshold + 1e-12), color="red", ls="--", lw=1.5, label=f"log10(threshold)")
axes[1].set_xlabel("log10(score)")
axes[1].set_ylabel("Density")
axes[1].set_title("分数分布（log10，便于看重叠）")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

try:
    from IPython.display import display
except ImportError:
    display = print

stats_df = pd.DataFrame(
    {
        "subset": ["Normal", "Attack"],
        "count": [int(normal_mask.sum()), int(attack_mask.sum())],
        "score_mean": [float(scores[normal_mask].mean()), float(scores[attack_mask].mean())],
        "score_median": [float(np.median(scores[normal_mask])), float(np.median(scores[attack_mask]))],
        "score_std": [float(scores[normal_mask].std()), float(scores[attack_mask].std())],
    }
)
display(stats_df.round(6))

In [ ]:
# ============ 训练集分位数 → 测试集 F1 敏感度（表格 + 曲线） ============
# 与主流程一致：若已做分数平滑，则在此用平滑后的 train/test 分数（不含段后处理）
_tr = train_scores_used if "train_scores_used" in globals() else train_point_scores
_te = test_scores_used if "test_scores_used" in globals() else test_point_scores
q_list = np.linspace(0.85, 0.99, 15)
rows = []
for q in q_list:
    th = float(np.quantile(_tr, q))
    pr = (_te > th).astype(int)
    p0, r0, f0 = prf1(test_labels, pr)
    pa = point_adjust(test_labels, pr.copy())
    p1, r1, f1 = prf1(test_labels, pa)
    rows.append(
        {
            "quantile": round(float(q), 4),
            "threshold": th,
            "Raw_P": p0,
            "Raw_R": r0,
            "Raw_F1": f0,
            "PA_P": p1,
            "PA_R": r1,
            "PA_F1": f1,
        }
    )

df_q = pd.DataFrame(rows)
display(df_q.round(4))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(df_q["quantile"], df_q["Raw_F1"], "o-", label="Raw F1")
ax.plot(df_q["quantile"], df_q["PA_F1"], "s-", label="PA F1")
ax.axvline(quantile, color="gray", ls="--", alpha=0.7, label=f"current q={quantile}")
ax.set_xlabel("Train score quantile (threshold)")
ax.set_ylabel("F1")
ax.set_title("分位数敏感度（越高阈值 → 通常 Recall↓ Precision↑）")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

best_raw = df_q.loc[df_q["Raw_F1"].idxmax()]
best_pa = df_q.loc[df_q["PA_F1"].idxmax()]
print("Raw F1 最大行:\n", best_raw.to_string())
print("\nPA F1 最大行:\n", best_pa.to_string())

In [ ]:
# ============ Raw vs Point-Adjust 指标柱状图 ============
fig, ax = plt.subplots(figsize=(7, 4))
metrics = ["Precision", "Recall", "F1"]
x = np.arange(len(metrics))
w = 0.35
bars0 = ax.bar(x - w / 2, [raw_p, raw_r, raw_f1], w, label="Raw", color="steelblue")
bars1 = ax.bar(x + w / 2, [pa_p, pa_r, pa_f1], w, label="Point-Adjust", color="darkorange")
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel("Score")
ax.set_ylim(0, 1.05)
ax.set_title("TranAD+CL：点级 vs Point-Adjust 对比")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

for bars in (bars0, bars1):
    for b in bars:
        h = b.get_height()
        ax.annotate(f"{h:.3f}", xy=(b.get_x() + b.get_width() / 2, h), xytext=(0, 2), textcoords="offset points", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ============ 局部时间窗：分数 + 阈值 + 真实攻击段 ============
# 自动选一段「含攻击较多」的窗口，便于肉眼看效果
seg_len = 8000
attack_idx = np.where(test_labels == 1)[0]
if len(attack_idx) > 0:
    center = attack_idx[len(attack_idx) // 4]
    t0 = max(0, int(center - seg_len // 2))
else:
    t0 = 10000
t1 = min(len(test_labels), t0 + seg_len)

tt = np.arange(t0, t1)
seg_scores = (test_scores_used if "test_scores_used" in globals() else test_point_scores)[t0:t1]
seg_true = test_labels[t0:t1]

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(tt, seg_scores, lw=0.8, alpha=0.85, label="anomaly score")
ax.axhline(threshold, color="red", ls="--", lw=1.2, label=f"threshold (q={quantile})")
# 真实攻击段浅红底
in_attack = seg_true == 1
if in_attack.any():
    y_min, y_max = seg_scores.min(), max(seg_scores.max(), threshold * 1.05)
    ax.fill_between(tt, y_min, y_max, where=in_attack, color="red", alpha=0.12, label="true attack")
ax.set_xlim(t0, t1 - 1)
ax.set_xlabel("time index (test)")
ax.set_ylabel("score")
ax.set_title(f"测试序列局部视图 [{t0}, {t1})")
ax.legend(loc="upper right")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============（可选）导出评估表到 CSV（默认保存在 Jupyter 当前工作目录） ============
df_summary.to_csv("tranad2_cl_eval_summary.csv", index=False, encoding="utf-8-sig")
df_q.to_csv("tranad2_cl_quantile_sweep.csv", index=False, encoding="utf-8-sig")
print("Saved: tranad2_cl_eval_summary.csv, tranad2_cl_quantile_sweep.csv")
print("绝对路径示例:", os.path.abspath("tranad2_cl_eval_summary.csv"))